# Configure Python Imports

In order to run our Jupyter Notebook,  we will need the following libraries:
- requests:  This library is used to make HTTP Requests to the TDP API
- json:  Allows us to manipulate files as JSON Objects
- pandas:  This is a very useful library for storing data in tabular structures
- numpy:  Open Source Framework for mathematical computation
- matplotlib:  Library for creating visualizations of your data

In [ ]:
import requests, json, pandas, numpy, matplotlib.pyplot as plt
%matplotlib inline

# Configure Connection Variables

Create and store information on how to connect to the TDP API. Set the following values in the code cell below, inside the empty quotation marks:

* Set `api_root` to point at the correct URL for your TDP environment
 * If your TDP instance URL is: https://platform.tetrascience.com
then your `api_root` should be: `"https://api.tetrascience.com"` - note that you do not need “platform” in this case, this is an exception.
 * If your TDP URL is anything other than “platform.tetrascience.com”, such as: https://platform-eu.tetrascience.com then your `api_root` should be the full hostname with "api" included: `"https://api.platform-eu.tetrascience.com"`
* Set `orgSlug` to be your org slug (ex: `training-jsmith`)
* Set `userToken` to an auth token, either your user's personal JWT or a Service User's token
* To get your user's JWT value, open or return to a Chrome tab with TDP, click on the main hamburger menu in the top left corner and select My Account. You should be presented with the Account details screen. Click the \"Copy Token\" button. Your clipboard will now have the token needed for calling the API.
* Set `userLabel` to match the value that you had entered in Lab 1 for your “user” Label. It should look similar to the following: [first_name]-[last_name]. E.g. if your name is John Smith, you would have used `john-smith`. Verify this value on a file or in the Agent configuration if you do not get results in later steps.


In [ ]:
# Example api_root:
# US Multitenant: "https://api.tetrascience.com" - note that "platform" is not included
# EU Multitenant: "https://api.platform-eu.tetrascience.com"
api_root = ""
orgSlug = ""
userToken = ""
userLabel = ""

In [ ]:
headers = {"x-org-slug": orgSlug, "ts-auth-token": userToken}
SearchURL = api_root + "/v1/datalake/searchEql"

SearchURL

# Set Query to Search for All Empower Projects

The TDP API uses ElasticSearch for indexing the Empower Content.  This powerful tool allows advanced searching against the data to find the appropriate information based upon your use case(s).

In this scenario, we are creating a query to find all of the Empower Data for the Project Names within.  We are using the "aggs" function of the ElasticSearch API to then collect all of the unique Project Names.

In [ ]:
payload = {
    "size": 0,
      "query": {
        "bool": {
          "must": [
            {
              "term": {"idsType": "lcuv-empower"}
            },
            {
              "nested": {
                "path": "labels",
                "query": {
                  "bool": {
                    "must": [
                      {"term": {"labels.name": "user"}},
                      {"term": {"labels.value": userLabel}}
                    ]
                  }
                }
              }
            }
          ]
        }
    },
    "aggs": {
        "unique_project_names": {
           "terms": { "field": "data.project.name", "size": 500 }
        }
    }
}

# Run Search Request and Display Results

We will now use the requests library to make a request to the TDP API.  We have previously configured the connection variables as well as the query we are executing.

Our final line just has the variable "result" in it which tells the Notebook to print the value of the variable.

In [ ]:
request = requests.post(SearchURL, json=payload, headers=headers)

result = request.json()

result

Now let's isolate the results by accessing data by attribute names and assign it to a Pandas DataFrame object for viewing the data as a table.  (We only uploaded one project to our instance and therefore there is only one result. Typically you'd see many results).

We are also renaming the columns in the DataFrame to make them more human readable.

In [ ]:
projects = result.get("aggregations").get("unique_project_names").get("buckets")

df = pandas.DataFrame(projects)
df = df.rename(columns={"key": 'Project Name', "doc_count": "No Injections"})

df

# Now Go Get All Injections for the Project and Display in a Table

First let's create the payload for the API Query.  We are retrieving values from the first row of the previous data table to pass in to the query.  The first value specifies the amount of results we desire.  The second one filters the query to only respond with injections associated with the Project Name in question.

Since this query will return all of the Empower Data Files (Injections) for this specific project, we would like to specify the fields that are relevant for our use case.

The "Fields" variable is being used for specifying the data of interest.  Notice how we use the json_normalize function to allow us to specify data in nested JSON objects (The injection data).  We are also renaming the JSON paths to field names that are more human readable in the DataFrame.

Finally you'll notice that we are using the pandas DataFrame fillna function.  This allows us to replace Null values with something of our choosing.  This will be relevant in the next exercise.

In [ ]:
payload_inj = {
    "size": int(df["No Injections"][0]),
    "query": {
        "bool": {
            "must": [
                {"nested": {
                    "path": "labels",
                    "query": {
                      "bool": {
                        "must": [
                          {"term": {"labels.name": "user"}},
                          {"term": {"labels.value": userLabel}}
                        ]
                      }
                    }
                  }
                },
                {"term": {"data.project.name": df["Project Name"][0]}}
            ]
        }
    }
}

request_inj = requests.post(SearchURL, json=payload_inj, headers=headers)
result_inj = request_inj.json()

allHits = result_inj.get("hits").get("hits")

df_inj_runs = pandas.json_normalize(
    allHits,
    ["_source", ["data", "runs"]],
    record_prefix='_source.data.runs.',
    meta=[["_source", "fileId"]])

df_inj_methods = pandas.json_normalize(
    allHits,
    ["_source", ["data", "methods"]]
    , record_prefix='_source.data.methods.')

df_inj_systems = pandas.json_normalize(
    allHits,
    ["_source", ["data", "systems"]]
    , record_prefix='_source.data.systems.')

df_inj = df_inj_runs.join(df_inj_methods).join(df_inj_systems)

Fields = ["_source.fileId"
          , "_source.data.runs.injection.id"
          , "_source.data.methods.sample_set.name"
          , "_source.data.runs.injection.time.acquisition"
          , "_source.data.runs.injection.volume.value"
          , "_source.data.runs.injection.volume.unit"
         ]

renamedDataFrame = df_inj[Fields].rename(
    columns={"_source.fileId": "File Id"
             , "_source.data.runs.injection.id": "Injection Id"
             , "_source.data.methods.sample_set.name": "Sample Set Name"
             , "_source.data.runs.injection.time.acquisition": "Acquisition Time"
             , "_source.data.runs.injection.volume.value": "Volume"
             , "_source.data.runs.injection.volume.unit": "Unit"})

renamedDataFrame["Sample Set Name"].fillna("Missing Value", inplace=True)

renamedDataFrame

# Display a Bar Chart Showing the Number of Injections per Sample Set

In order to display this bar chart, we will need to isolate the unique Sample Set Names in our data set and assign them to an array.  This also is required to get the Total Number of Injections per Sample Set Name.   Both of these arrays will be used as input for the Bar Chart.

We are using an Open Source Python Framework called MatPlotLib for showing the data visualization.

In [ ]:
sampleSets = renamedDataFrame["Sample Set Name"].unique()
print("Sample Sets As Array:")
print(sampleSets)

sampleSetInjectionCounts = renamedDataFrame.groupby("Sample Set Name").count()["Injection Id"].to_numpy()
print("\nInjection Counts As Array:")
print(sampleSetInjectionCounts)

Now let's configure the Horizontal Bar Chart and plot the chart on the Notebook.

In [ ]:
fig, ax = plt.subplots();

ax.barh(sampleSets, sampleSetInjectionCounts)
ax.set_title("Injections Per Sample Set")
ax.set_xlabel("Injections")

plt.show()

# Sort Data by System Utilization

For our final use case, let's group the Empower Data based upon overall System Run Time. This information can be useful in ensuring your lab personnel balances their runs across all of their equipment which of course can help maximize investment.

We are going to work with the existing data set we have and will query the specific fields of interest.  We are looking for the Injection Run Time (Both the units - Minutes, Hours, etc... and value).  Once we have the DataFrame with the data of interest, we can group the data set by System Name and Run Time Unit.   The new DataFrame can then be displayed and ordered by Run Time Unit and Run Time in a descending manner.

In [ ]:
Fields = ["_source.data.systems.name"
          , "_source.data.runs.injection.run_time.value"
          , "_source.data.runs.injection.run_time.unit"]

renamedDataFrame = df_inj[Fields].rename(
    columns={"_source.data.systems.name": "System Name"
             , "_source.data.runs.injection.run_time.value": "Run Time"
             , "_source.data.runs.injection.run_time.unit": "Run Time Unit"})


renamedDataFrame = renamedDataFrame.groupby(["System Name", "Run Time Unit"]).sum()

renamedDataFrame.sort_values(by=["Run Time Unit", "Run Time"], ascending=False)